# Leson 11 - Model Context Protocol (MCP)

Di **Model Context Protocol (MCP)** na open standard wey enable agents make dem fit dynamically discover and use tools, resources, and data sources for runtime. Instead of hardcoding tools into an agent, MCP dey allow agents connect to external servers wey dey expose capabilities on demand.

For dis leson, you go learn:
- Wetin MCP be and why e matter for agent systems
- How di MCP client-server architecture dey work
- How to build agents wey dey use MCP-style tool discovery


## How to set am up

**Wetin you need:**
- Azure AI Foundry project wey don deploy model
- Run `az login` make you fit authenticate


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be the Model Context Protocol (MCP)?

MCP dey define one standard way for AI agents to discover and interact with external tools and data sources:

- **MCP Server**: E dey expose tools, resources, and prompts via a standard protocol
- **MCP Client**: Na the agent runtime wey dey connect to servers and dey discover available capabilities
- **Dynamic Discovery**: Agents no need hardcoded tools — dem go discover wetin dey available at runtime

Dis dey powerful for building extensible agent systems wey new capabilities fit add without modifying the agent code.


## How MCP Dey Work

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Di agent (MCP client) dey connect to an MCP server
2. Di server dey respond wit a list of available tools and dem schemas
3. Di agent fit den call any tool wey e don find while e dey reason
4. Results dey flow back through di same protocol


## Simulate How Dem Dey Discover MCP Tools

As real MCP server need one running server process, we go demonstrate the pattern by using `@tool` functions wey dey simulate wetin an MCP-connected accommodation service go provide.

For production, these tools dem go dey discovered dynamically from an MCP server instead of being defined locally.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## How to build one agent wit MCP-style tools


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP for Production

For production environment, MCP dey enable strong patterns:

- **Dynamic tool discovery**: Agents dey connect to MCP servers and dem go find tools while dem dey run
- **Decoupled architecture**: Tool providers fit update on dia own without touching the agent
- **Cross-organization sharing**: Teams fit expose capabilities via MCP servers wey any agent fit use
- **Microsoft Agent Framework support**: MAF get built-in MCP client support via the `mcp` integration

If you wan use real MCP server with MAF, you go connect via `hosted_mcp_tool()` or the MCP client integration.

**Find out more:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

For dis lesson, you don learn:
- **MCP** na open standard wey dey allow dynamic discovery of tools between agents and tool providers
- The **client-server architecture** dey make agents fit discover capabilities during runtime
- MCP dey enable **extensible, decoupled agent systems** wey allow tools to dey added without any code change
- Microsoft Agent Framework dey provide **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dis document na AI translate with di translation service Co-op Translator (https://github.com/Azure/co-op-translator). Even though we dey try make am correct, abeg note say automated translations fit get mistakes or no too accurate. Di original document for im own language na di one wey get final authority. If na important information, make you use professional human translator. We no go responsible for any misunderstanding or wrong interpretation wey fit arise from using this translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
